# Day 2 

### Track A - Prepare the text

In [4]:
from datasets import load_dataset

ds = load_dataset("mediabiasgroup/mbib-base")
gender_ds = ds["gender_bias"]

df = gender_ds.to_pandas()

df.head()

,id,text,label,dataset_id
0,107-9295,MENTION4637 I am so sorry that you re in this ...,0,107
1,107-10267,Being a Boomer is Terribly Difficult Task Sinc...,0,107
2,107-11607,MENTION3217 Big accomplishments west point and...,0,107
3,107-1052,I ve never met a kid who can write poetry,0,107
4,107-10961,Talk the talk but can t walk the walk MKR,0,107


In [5]:
df = df[df["text"].astype(str).str.strip() != ""].reset_index(drop=True)
print(df.shape)

(17939, 4)


In [6]:
# Keep original dataset unchanged
clean_df = df.copy()

# Normalize text for duplicate detection
clean_df["normalized_text"] = (
    clean_df["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identify conflicting-label groups
label_counts = (
    clean_df.groupby("normalized_text")["label"]
    .nunique()
)

conflicting_keys = label_counts[
    label_counts > 1
].index

# Remove all conflicting-label examples
clean_df = clean_df[
    ~clean_df["normalized_text"].isin(conflicting_keys)
].copy()

# Keep one example per consistent duplicate group
clean_df = clean_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

print("Original:", len(df))
print("Cleaned:", len(clean_df))
print("Removed:", len(df) - len(clean_df))

df = clean_df.drop(columns=["normalized_text"]).reset_index(drop=True)
print("final dataset:", len(df))

Original: 17939
Cleaned: 17893
Removed: 46
final dataset: 17893


In [7]:
# --- 0. Setup ---------------------------------------------------------------


import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             precision_score, recall_score)

SEED = 42
np.random.seed(SEED)


In [8]:
# --- 1. Split: 80 / 10 / 10, stratified -------------------------------------

X_temp, X_test, y_temp, y_test = train_test_split(
    df["text"], df["label"], test_size=0.10, stratify=df["label"], random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=1/9, stratify=y_temp, random_state=SEED
)

print(f"train {len(X_train)} ({len(X_train)/len(df):.0%}) | "
      f"val {len(X_val)} ({len(X_val)/len(df):.0%}) | "
      f"test {len(X_test)} ({len(X_test)/len(df):.0%})")
print(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).round(3),
    "val":   y_val.value_counts(normalize=True).round(3),
    "test":  y_test.value_counts(normalize=True).round(3),
}))

train 14313 (80%) | val 1790 (10%) | test 1790 (10%)
       train    val   test
label                     
0      0.501  0.501  0.501
1      0.499  0.499  0.499


In [6]:
import re
# --- 2. Leakage checks ------------------------------------------------------

print("Exact overlap train∩val :", len(set(X_train) & set(X_val)))
print("Exact overlap train∩test:", len(set(X_train) & set(X_test)))

norm = lambda s: re.sub(r"\s+", " ", str(s).lower().strip())
print("Normalized train∩val    :", len(set(map(norm, X_train)) & set(map(norm, X_val))))

print("\nAuthor identifiers in text:")
for name, pat in {"@handles": r"@\w+", "URLs": r"http\S+", "emails": r"\S+@\S+\.\S+"}.items():
    print(f"  {name:<10} {df['text'].str.contains(pat, na=False).sum()}")

print("\nSource distribution across splits:")
print(pd.DataFrame({
    "train": df.loc[X_train.index, "dataset_id"].value_counts(normalize=True),
    "val":   df.loc[X_val.index, "dataset_id"].value_counts(normalize=True),
    "test":  df.loc[X_test.index, "dataset_id"].value_counts(normalize=True),
}).round(3))

Exact overlap train∩val : 0
Exact overlap train∩test: 0
Normalized train∩val    : 0

Author identifiers in text:
  @handles   0
  URLs       0
  emails     0

Source distribution across splits:
            train    val   test
dataset_id                     
75          0.510  0.530  0.522
107         0.442  0.415  0.422
106         0.049  0.055  0.056


In [9]:
# --- 5. Text preparation: tokenizer -----------------------------------------

# Make the stable Gender tokenizer importable when the model is loaded later.
from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "models").is_dir() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the BiasLens folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import tokenize

for t in X_train.head(3):
    print(f"raw   : {t[:90]}")
    print(f"tokens: {tokenize(t)[:12]}  (len {len(tokenize(t))})\n")


 # --- 6. Vocabulary — fitted on TRAIN only -----------------------------------

MAX_VOCAB = 20000

def build_vocab(texts, min_freq=2, max_size=MAX_VOCAB, specials=("<pad>", "<unk>")):
    counter = Counter(tok for t in texts for tok in tokenize(t))
    kept = [w for w, c in counter.most_common() if c >= min_freq][:max_size - len(specials)]
    itos = list(specials) + kept
    return {w: i for i, w in enumerate(itos)}, itos, counter

stoi, itos, counter = build_vocab(X_train)
PAD_IDX, UNK_IDX = stoi["<pad>"], stoi["<unk>"]

print("unique tokens in train:", len(counter))
print("vocabulary size       :", len(stoi))
print("token coverage        :", round(
    sum(c for w, c in counter.items() if w in stoi) / sum(counter.values()), 4))

# --- 7. Max sequence length — from TRAIN only -------------------------------

lens = X_train.map(lambda t: len(tokenize(t)))
print(lens.describe().round(1))
for q in [0.90, 0.95, 0.99]:
    print(f"p{int(q*100)}: {lens.quantile(q):.0f}")

MAX_LEN = int(lens.quantile(0.95))
print(f"\nMAX_LEN = {MAX_LEN} (covers {(lens <= MAX_LEN).mean():.1%} untruncated)")


# --- 8. Encode + pad --------------------------------------------------------

def encode(text, stoi=stoi, max_len=None):
    max_len = max_len or MAX_LEN
    ids = [stoi.get(tok, UNK_IDX) for tok in tokenize(text)][:max_len]
    return ids + [PAD_IDX] * (max_len - len(ids))

def encode_series(texts, stoi=stoi, max_len=None):
    return np.array([encode(t, stoi, max_len) for t in texts], dtype=np.int64)

X_train_ids = encode_series(X_train)
X_val_ids   = encode_series(X_val)
X_test_ids  = encode_series(X_test)

print("shapes:", X_train_ids.shape, X_val_ids.shape, X_test_ids.shape)



# --- 9. Inspect tokenized examples ------------------------------------------

for i in range(3):
    ids = X_train_ids[i]
    print(f"raw    : {X_train.iloc[i][:80]}")
    print(f"ids    : {ids[:12]} ...")
    print(f"decoded: {[itos[j] for j in ids][:12]} ...")
    print(f"pads   : {(ids == PAD_IDX).sum()}/{len(ids)}  label: {y_train.iloc[i]}\n")

print("<unk> rate train:", round((X_train_ids == UNK_IDX).sum() / (X_train_ids != PAD_IDX).sum(), 4))
print("<unk> rate val  :", round((X_val_ids == UNK_IDX).sum() / (X_val_ids != PAD_IDX).sum(), 4))

# --- 10. Label encoding -----------------------------------------------------

print("label values:", sorted(df["label"].unique()), "| dtype:", df["label"].dtype)

y_train_enc = y_train.to_numpy(dtype=np.int64)
y_val_enc   = y_val.to_numpy(dtype=np.int64)
y_test_enc  = y_test.to_numpy(dtype=np.int64)

LABEL_NAMES = {0: "no bias", 1: "bias"}

# sanity: labels still aligned with their texts
assert len(X_train_ids) == len(y_train_enc)
print("alignment check passed")

# --- 11. Reusable preparation class -----------------------------------------

class TextPrep:
    """Fit on training text only; transform any split."""

    def __init__(self, min_freq=2, max_vocab=MAX_VOCAB, len_quantile=0.95):
        self.min_freq, self.max_vocab, self.len_quantile = min_freq, max_vocab, len_quantile

    def fit(self, texts):
        self.stoi, self.itos, self.counter = build_vocab(texts, self.min_freq, self.max_vocab)
        self.max_len = int(pd.Series([len(tokenize(t)) for t in texts]).quantile(self.len_quantile))
        self.pad_idx, self.unk_idx = self.stoi["<pad>"], self.stoi["<unk>"]
        return self

    def transform(self, texts):
        return encode_series(texts, self.stoi, self.max_len)

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

prep = TextPrep().fit(X_train)
print("vocab:", len(prep.stoi), "| max_len:", prep.max_len)

raw   : If you want something new you have to stop doing something old
tokens: ['if', 'you', 'want', 'something', 'new', 'you', 'have', 'to', 'stop', 'doing', 'something', 'old']  (len 12)

raw   : haha silly we know all bisexuals are just confused s but ive legit had people say that to 
tokens: ['haha', 'silly', 'we', 'know', 'all', 'bisexuals', 'are', 'just', 'confused', 's', 'but', 'ive']  (len 19)

raw   : yes black people are terrible sinners i remember reading about how you enslaved white peop
tokens: ['yes', 'black', 'people', 'are', 'terrible', 'sinners', 'i', 'remember', 'reading', 'about', 'how', 'you']  (len 25)

unique tokens in train: 15057
vocabulary size       : 7626
token coverage        : 0.968
count    14313.0
mean        16.2
std          7.1
min          1.0
25%         11.0
50%         16.0
75%         22.0
max         48.0
Name: text, dtype: float64
p90: 26
p95: 28
p99: 33

MAX_LEN = 28 (covers 96.4% untruncated)
shapes: (14313, 28) (1790, 28) (1790, 28)
raw    : 

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

# --- Baseline 1: Dummy Classifier -------------------------------------------

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

pred_dummy  = dummy.predict(X_val)
proba_dummy = dummy.predict_proba(X_val)[:, 1]

print("Dummy — Validation")
print(f"  F1        : {f1_score(y_val, pred_dummy, zero_division=0):.4f}")
print(f"  Precision : {precision_score(y_val, pred_dummy, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_val, pred_dummy, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_val, proba_dummy):.4f}")
print("  Confusion matrix [[TN FP][FN TP]]:")
print(confusion_matrix(y_val, pred_dummy))

Dummy — Validation
  F1        : 0.0000
  Precision : 0.0000
  Recall    : 0.0000
  ROC-AUC   : 0.5000
  Confusion matrix [[TN FP][FN TP]]:
[[896   0]
 [894   0]]


In [9]:
# --- Baseline 2: TF-IDF + Logistic Regression -------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SCORING = ["f1", "precision", "recall", "roc_auc"]



tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None,max_features=20000,
                              ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

# 3-Fold CV — TF-IDF refitted inside each fold
cv_lr = cross_validate(tfidf_lr, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)

print("TF-IDF + LogReg — 3-Fold CV (train)")
for m in SCORING:
    s = cv_lr[f"test_{m}"]
    print(f"  {m:<10} {s.mean():.4f} (+/- {s.std():.4f})")

# Validation
tfidf_lr.fit(X_train, y_train)
pred_lr  = tfidf_lr.predict(X_val)
proba_lr = tfidf_lr.predict_proba(X_val)[:, 1]

print("\nTF-IDF + LogReg — Validation")
print(f"  F1        : {f1_score(y_val, pred_lr):.4f}")
print(f"  Precision : {precision_score(y_val, pred_lr):.4f}")
print(f"  Recall    : {recall_score(y_val, pred_lr):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_val, proba_lr):.4f}")
print("  Confusion matrix [[TN FP][FN TP]]:")
print(confusion_matrix(y_val, pred_lr))

print(f"\n  features: {len(tfidf_lr.named_steps['tfidf'].vocabulary_)}")

TF-IDF + LogReg — 3-Fold CV (train)
  f1         0.8169 (+/- 0.0040)
  precision  0.7790 (+/- 0.0040)
  recall     0.8587 (+/- 0.0043)
  roc_auc    0.8893 (+/- 0.0025)

TF-IDF + LogReg — Validation
  F1        : 0.8274
  Precision : 0.7877
  Recall    : 0.8714
  ROC-AUC   : 0.8989
  Confusion matrix [[TN FP][FN TP]]:
[[686 210]
 [115 779]]

  features: 20000


### Key Findings — Baseline Experiment

- TF-IDF + Logistic Regression reached a validation **F1 of 0.8286**, with **ROC-AUC of 0.8994**.
- 3-fold cross-validation on the training data gave **F1 = 0.8177 (± 0.0039)**, confirming a stable fit with no sign of overfitting.
- `DummyClassifier(most_frequent)` returned F1 = 0.000, but the meaningful floor is **F1 = 0.667** — the score of a constant predictor choosing the positive class. The model's real gain over a text-blind baseline is **+0.16 F1**.
- Precision (0.7881) trails recall (0.8736): 210 false positives against 113 false negatives. The model over-predicts bias.
.


In [10]:
src = df.loc[X_val.index, "dataset_id"]

for d in sorted(src.unique()):
    m = (src == d).to_numpy()
    print(f"dataset {d}: n={m.sum():<5} "
          f"F1={f1_score(y_val[m], pred_lr[m], zero_division=0):.4f}  "
          f"acc={(y_val[m].to_numpy() == pred_lr[m]).mean():.4f}")

dataset 75: n=949   F1=0.8550  acc=0.7766
dataset 106: n=98    F1=0.7903  acc=0.7347
dataset 107: n=743   F1=0.7071  acc=0.8829


- Per-source evaluation on the validation set reveals uneven performance: F1 = 0.856 on source 75, 0.790 on source 106, but only **0.713 on source 107** — a 0.14 spread. Source 107 simultaneously shows the *highest* accuracy (0.884) and the lowest F1, because it is dominated by negative examples: the model scores easy wins on the majority class while missing most of its positive cases. This confirms the model partially exploits source-level distribution rather than detecting bias from content alone.

In [11]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(SEED)

VOCAB_SIZE = len(stoi)
EMBED_DIM  = 64
LSTM_UNITS = 32

# sanity checks
assert X_train_ids.shape == (len(y_train_enc), MAX_LEN)
assert X_val_ids.shape   == (len(y_val_enc),   MAX_LEN)
assert PAD_IDX == 0, "mask_zero=True requires the pad index to be 0"
assert X_train_ids.max() < VOCAB_SIZE

print(f"vocab={VOCAB_SIZE} | max_len={MAX_LEN}")
print(f"train {X_train_ids.shape} | val {X_val_ids.shape}")
print(f"padding is post: {(X_train_ids[0] == PAD_IDX).argmax() > 0}")
print(f"label check — first train text: {X_train.iloc[0][:60]!r} -> {y_train_enc[0]}")

vocab=7626 | max_len=28
train (14313, 28) | val (1790, 28)
padding is post: True
label check — first train text: 'If you want something new you have to stop doing something o' -> 0


In [12]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
              mask_zero=True, name="embedding"),
    LSTM(LSTM_UNITS, name="lstm"),
    Dense(1, activation="sigmoid", name="output"),
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 28, 64)         │       488,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,513 (1.91 MB)

 Trainable params: 500,513 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stop = EarlyStopping(monitor="val_loss", patience=2,
                           restore_best_weights=True, verbose=1)

history = model.fit(
    X_train_ids, y_train_enc,
    validation_data=(X_val_ids, y_val_enc),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)

print(f"\nstopped at epoch {len(history.history['loss'])} | "
      f"best val_loss {min(history.history['val_loss']):.4f}")

Epoch 1/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7499 - loss: 0.5083 - val_accuracy: 0.8095 - val_loss: 0.4295
Epoch 2/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8451 - loss: 0.3555 - val_accuracy: 0.8117 - val_loss: 0.4410
Epoch 3/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8849 - loss: 0.2862 - val_accuracy: 0.8061 - val_loss: 0.4803
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.

stopped at epoch 3 | best val_loss 0.4295


In [14]:
proba_lstm = model.predict(X_val_ids, verbose=0).ravel()
pred_lstm  = (proba_lstm >= 0.5).astype(int)

print("LSTM — Validation")
print(f"  F1        : {f1_score(y_val_enc, pred_lstm):.4f}")
print(f"  Precision : {precision_score(y_val_enc, pred_lstm, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_val_enc, pred_lstm):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_val_enc, proba_lstm):.4f}")
print("  Confusion matrix [[TN FP][FN TP]]:")
print(confusion_matrix(y_val_enc, pred_lstm))

LSTM — Validation
  F1        : 0.8219
  Precision : 0.7708
  Recall    : 0.8803
  ROC-AUC   : 0.8777
  Confusion matrix [[TN FP][FN TP]]:
[[662 234]
 [107 787]]


In [15]:
src = df.loc[X_val.index, "dataset_id"]

for d in sorted(src.unique()):
    m = (src == d).to_numpy()
    print(f"dataset {d}: n={m.sum():<5} "
          f"F1={f1_score(y_val_enc[m], pred_lstm[m], zero_division=0):.4f}  "
          f"acc={(y_val_enc[m] == pred_lstm[m]).mean():.4f}")

dataset 75: n=949   F1=0.8481  acc=0.7671
dataset 106: n=98    F1=0.8406  acc=0.7755
dataset 107: n=743   F1=0.6957  acc=0.8681


### Model Comparison

Primary metric: **ROC-AUC**. Supporting: F1, precision, recall.

| Model | ROC-AUC | F1 | Precision | Recall |
|---|---|---|---|---|
| LSTM (Embedding → LSTM → Dense) | 0.8777 | 0.8219 | 0.7708 | **0.8803** |
| **TF-IDF + Logistic Regression** | **0.8994** | **0.8286** | **0.7881** | 0.8736 |

All figures on the validation set (1,790 rows). The LSTM edges ahead on recall alone; the Logistic Regression leads on the primary metric by 0.0217 and on every other measure.

ROC-AUC is threshold-independent, so the gap reflects genuinely weaker ranking ability in the LSTM rather than a poorly calibrated decision boundary.

**Selected: TF-IDF + Logistic Regression.**

In [16]:
# --- Error analysis — validation set ---------------------------------------

import numpy as np

err = pd.DataFrame({
    "text":  X_val.values,
    "true":  y_val.values,
    "proba": proba_lr,
    "pred":  pred_lr,
    "source": df.loc[X_val.index, "dataset_id"].values,
})
err["correct"] = err["true"] == err["pred"]

print(f"errors: {(~err.correct).sum()} / {len(err)}  ({(~err.correct).mean():.1%})\n")

# Most confident mistakes
fp = err[(err.true == 0) & (err.pred == 1)].nlargest(5, "proba")
fn = err[(err.true == 1) & (err.pred == 0)].nsmallest(5, "proba")

print("=" * 70)
print("FALSE POSITIVES — labelled 'no bias', model was most sure it was bias")
print("=" * 70)
for _, r in fp.iterrows():
    print(f"[p={r.proba:.3f} src={r.source}] {r.text[:95]}")

print("\n" + "=" * 70)
print("FALSE NEGATIVES — labelled 'bias', model was most sure it was not")
print("=" * 70)
for _, r in fn.iterrows():
    print(f"[p={r.proba:.3f} src={r.source}] {r.text[:95]}")

# Where the model hesitates
unsure = err[(err.proba > 0.4) & (err.proba < 0.6)]
print(f"\n\nuncertain predictions (0.4 < p < 0.6): {len(unsure)} "
      f"({len(unsure)/len(err):.1%}), accuracy there: {unsure.correct.mean():.1%}")

errors: 325 / 1790  (18.2%)

FALSE POSITIVES — labelled 'no bias', model was most sure it was bias
[p=0.950 src=75] girl im not a nurse yet slices hand same girl is that blood well she isnt a nurse yet 
[p=0.927 src=75] jews are the devil dc
[p=0.909 src=75] i teach art in college and my wife is a middle school teacher 
[p=0.904 src=75] the bible says being gay is a sin gays will burn in hell we should try and help them mfw yeah g
[p=0.898 src=75] whites are a bigger threat to their in group in a a majority white country like america than mu

FALSE NEGATIVES — labelled 'bias', model was most sure it was not
[p=0.129 src=107] I m really wondering by there are 3 chicks and 1 dude hosting ESPN s college football show Am I
[p=0.146 src=107]  We re not just the dumb blondes with the pretty faces you re certainly not the second part any
[p=0.149 src=107] MENTION786 Elegant and beautiful Cheap and trashy Nothing more unattractive than girls banging 
[p=0.163 src=106] Ladies Seriously Stop put

### Error Analysis — Validation Set

323 errors out of 1,790 predictions (18.0%).

**Label noise, not model error.** Among the five highest-confidence false positives, three contain explicitly biased content yet carry label 0 — the model was correct and was penalised for it. Only one is a genuine failure: a neutral sentence about teaching misclassified at p=0.908, likely driven by gendered tokens ("wife", "teacher") that correlate with the positive class in training. This confirms the annotation noise identified during cleaning and places a ceiling on achievable performance.

**Implicit bias escapes the model.** All five highest-confidence false negatives express bias through sarcasm, framing or implicit assumption rather than explicit vocabulary. With no lexical trigger to key on, a bag-of-words representation has nothing to detect. Four of the five originate from source 107, directly explaining that source's depressed F1 (0.713): its positive cases are disproportionately subtle.

**A fifth of predictions are genuine uncertainty.** 352 predictions (19.7%) fall between p=0.4 and p=0.6, with accuracy of 58.0% in that band — marginally above chance. These are cases the model cannot resolve, not threshold artefacts; adjusting the decision boundary will not recover them.

### 4.3 Save Model Pipeline

In [17]:
import joblib
from pathlib import Path

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models").is_dir() and (p / "notebooks").is_dir()),
    None
)
if project_root is None:
    raise RuntimeError("Run this notebook from inside the BiasLens project.")

model_path = project_root / "models" / "gender_bias_lr.joblib"
# The selected LR pipeline is the official production Gender model.
joblib.dump(tfidf_lr, model_path)
print("Model saved to:", model_path)


Model saved to: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/gender_bias_lr.joblib


### 4.4 Model Loading & Prediction

In [18]:
import numpy as np

loaded_model = joblib.load(model_path)

test_inputs = X_val.iloc[:5].tolist()

original_probs = tfidf_lr.predict_proba(test_inputs)
loaded_probs   = loaded_model.predict_proba(test_inputs)

np.testing.assert_allclose(original_probs, loaded_probs)

print("Model loading test passed.")

Model loading test passed.


In [19]:
def predict_bias(text, verbose=False):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide a non-empty text.")

    probability = float(loaded_model.predict_proba([text])[0, 1])
    label = "Biased" if probability >= 0.5 else "Unbiased"

    if not verbose:
        return label

    return {
        "bias_type": "gender",
        "predicted_label": int(probability >= 0.5),
        "prediction": label,
        "bias_probability": round(probability, 4),
    }

In [20]:
print(predict_bias("women are terrible drivers"))
print(predict_bias("the weather is nice today"))


Biased
Unbiased


## Final Model Evaluation — Test Set

Evaluate the saved Logistic Regression pipeline on the held-out
10% test split, using the original data preparation and random seed.

No model training, tuning, or artifact modification is performed.

In [10]:

# Final Gender Bias Test Evaluation
# Saved model only — NO training or model modification

from pathlib import Path
import sys
import joblib
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Locate BiasLens project
project_root = next(
    (
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "models").is_dir()
        and (p / "notebooks").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("BiasLens project folder not found.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Required for loading the saved Gender tokenizer
from src.preprocessing import tokenize

# Verify the original test split is available
assert len(df) == 17893, "Original cleaned dataset mismatch."
assert len(X_test) == len(y_test) == 1790, "Test split mismatch."

# Load the existing trained model
model_path = project_root / "models" / "gender_bias_lr.joblib"
model = joblib.load(model_path)

# Predict — no fit(), no training
positive_idx = list(model.classes_).index(1)
y_score = model.predict_proba(X_test)[:, positive_idx]
y_pred = (y_score >= 0.5).astype(int)

# Final Test Metrics
print("=== Gender Bias — Final Test Evaluation ===")
print(f"Test Samples : {len(X_test)}")
print(f"ROC-AUC      : {roc_auc_score(y_test, y_score):.4f}")
print(f"Accuracy     : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision    : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall       : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score     : {f1_score(y_test, y_pred, zero_division=0):.4f}")

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


=== Gender Bias — Final Test Evaluation ===
Test Samples : 1790
ROC-AUC      : 0.8695
Accuracy     : 0.7855
Precision    : 0.7540
Recall       : 0.8468
F1-score     : 0.7977

Confusion Matrix [[TN, FP], [FN, TP]]:
[[649 247]
 [137 757]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.72      0.77       896
           1       0.75      0.85      0.80       894

    accuracy                           0.79      1790
   macro avg       0.79      0.79      0.78      1790
weighted avg       0.79      0.79      0.78      1790

